# 🧬 Bio-JEPA — 3ème cible : SARS-CoV-2 Mpro (ChEMBL325)
**Objectif** : Valider la généralisation de Bio-JEPA sur une 3ème cible biologique

---

## ⚙️ Avant de commencer
1. `Runtime → Change runtime type → A100 GPU`
2. Avoir les checkpoints sur Google Drive

| Étape | Description | Durée |
|---|---|---|
| 0 | Setup GPU + Drive + Repo | 5 min |
| 1 | Few-shot Mpro avec checkpoint 100ep | ~30 min |
| 2 | Tableau comparatif A2A / EGFR / Mpro | 5 min |
| 3 | Sauvegarde Drive | 2 min |

---
## Étape 0 — Setup

In [1]:
import torch
!nvidia-smi
print(f'\n✓ GPU : {torch.cuda.get_device_name(0)}')
print(f'✓ VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Wed Mar 11 13:33:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/Bio-JEPA-checkpoints'
DRIVE_RES  = '/content/drive/MyDrive/Bio-JEPA-results'
os.makedirs(DRIVE_RES, exist_ok=True)
print('✓ Drive monté')

Mounted at /content/drive
✓ Drive monté


In [3]:
!pip install torch_geometric rdkit pandas numpy scikit-learn tqdm requests pyyaml chembl-webresource-client -q
print('✓ Dépendances installées')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.6 MB/s eta 0:00:00
✓ Dépendances installées


In [4]:
if not os.path.exists('/content/Bio-JEPA'):
    !git clone https://github.com/7Nayy/Bio-JEPA.git /content/Bio-JEPA
else:
    !git -C /content/Bio-JEPA pull origin main

os.chdir('/content/Bio-JEPA')
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results', exist_ok=True)

shutil.copy(f'{DRIVE_CKPT}/zinc_pretrained.pt', 'checkpoints/zinc_pretrained.pt')

# Récupérer aussi les résultats A2A et EGFR depuis Drive
shutil.copy(f'{DRIVE_RES}/few_shot_transfer.json', 'results/few_shot_transfer.json')
shutil.copy(f'{DRIVE_RES}/few_shot_egfr.json', 'results/few_shot_egfr.json')

print('✓ Repo prêt + checkpoints + résultats précédents récupérés')

Cloning into '/content/Bio-JEPA'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 123 (delta 22), reused 50 (delta 13), pack-reused 54 (from 2)
Receiving objects: 100% (123/123), 48.14 MiB | 37.29 MiB/s, done.
Resolving deltas: 100% (25/25), done.
✓ Repo prêt + checkpoints + résultats précédents récupérés


---
## Étape 1 — Few-shot sur SARS-CoV-2 Mpro (ChEMBL325)

> **SARS-CoV-2 Mpro** (Main Protease, ChEMBL325) — cible antivirale clé,
> structurellement très différente de A2A et EGFR


In [10]:
# ============================================================
# CELLULE VÉRIFICATION — Confirmer que c'est bien Mpro/ChEMBL325
# ============================================================
import os, torch, requests, pandas as pd
from pathlib import Path
from data.mol_graph import smiles_vers_graphe

CHEMBL_CACHE_MPRO = 'data/chembl325_graphs.pt'
Path('data').mkdir(exist_ok=True)

# --- Téléchargement si absent ---
if not os.path.exists(CHEMBL_CACHE_MPRO):
    print('⚠ Cache Mpro absent — téléchargement ChEMBL325...')
    BASE_ROOT = 'https://www.ebi.ac.uk'
    records = []
    url = BASE_ROOT + '/chembl/api/data/activity.json?target_chembl_id=CHEMBL325&pchembl_value__isnull=false&limit=500&offset=0'
    page = 0
    while url:
        r = requests.get(url, timeout=60, headers={'Accept': 'application/json'})
        if r.status_code != 200:
            print(f'⚠ HTTP {r.status_code}')
            break
        data = r.json()
        for act in data.get('activities', []):
            smi = act.get('canonical_smiles')
            val = act.get('pchembl_value')
            if smi and val:
                try:
                    records.append({'smiles': smi, 'pchembl_value': float(val)})
                except (ValueError, TypeError):
                    pass
        next_path = data.get('page_meta', {}).get('next')
        url = (BASE_ROOT + next_path) if next_path else None
        page += 1
        print(f'  Page {page} — {len(records)} activités...', end='\r')

    df = pd.DataFrame(records).drop_duplicates('smiles')
    print(f'\n✓ {len(df)} molécules ChEMBL325')

    data_list = []
    for _, row in df.iterrows():
        try:
            g = smiles_vers_graphe(row['smiles'])
            if g is not None:
                g.y = torch.tensor([row['pchembl_value']], dtype=torch.float)
                data_list.append(g)
        except Exception:
            pass

    torch.save(data_list, CHEMBL_CACHE_MPRO)
    import shutil
    shutil.copy(CHEMBL_CACHE_MPRO, '/content/drive/MyDrive/Bio-JEPA-checkpoints/chembl325_graphs.pt')
    print(f'✓ {len(data_list):,} graphes sauvegardés')

# --- Chargement et vérification ---
# APRÈS (fix minimal — Mpro only)
mpro_data   = torch.load(CHEMBL_CACHE_MPRO, weights_only=False)
labels_mpro = torch.tensor([d.y.item() for d in mpro_data])

print(f'\nChEMBL325 (Mpro) — N={len(mpro_data):,} | mean={labels_mpro.mean():.3f} | std={labels_mpro.std():.3f}')

print(f'\nChEMBL325 (Mpro) — N={len(mpro_data):,} | mean={labels_mpro.mean():.3f} | std={labels_mpro.std():.3f}')
print(f'ChEMBL251 (A2A)  — N={len(chembl_data):,} | mean={labels_a2a.mean():.3f} | std={labels_a2a.std():.3f}')

# Diagnostic bug
same_N = len(mpro_data) == len(chembl_data)
same_mean = abs(labels_mpro.mean() - labels_a2a.mean()) < 0.01

if same_N and same_mean:
    print('\n⛔ MÊME DATASET CHARGÉ — le few-shot Mpro est invalide, relancer avec chembl325_graphs.pt')
else:
    print('\n✓ Datasets distincts — few-shot Mpro valide')


ChEMBL325 (Mpro) — N=8,000 | mean=6.614 | std=1.128

ChEMBL325 (Mpro) — N=8,000 | mean=6.614 | std=1.128


NameError: name 'chembl_data' is not defined

In [5]:
# Few-shot avec checkpoint 100 époques
!python few_shot_eval.py \
    --checkpoint checkpoints/zinc_pretrained.pt \
    --target CHEMBL325 \
    --n-values 10,50,100,200,500,1000 \
    --n-runs 5 \
    --save-json results/few_shot_mpro.json

print('\n✓ Few-shot Mpro terminé')

  Évaluation Few-Shot : Bio-JEPA vs GNN supervisé
  Cible : CHEMBL325 — CHEMBL325
  N ∈ [10, 50, 100, 200, 500, 1000]
  5 runs par N  │  Dispositif : cuda

[Données] Chargement du dataset ChEMBL — target=CHEMBL325
  Matérialisation du train set en liste...

  Train total : 6,739  │  Val : 842  │  Test : 843

[Bio-JEPA] Checkpoint : checkpoints/zinc_pretrained.pt

[Bio-JEPA] Extraction des embeddings (Target Encoder figé)...
  Train : (6739, 256)  │  Val : (842, 256)  │  Test : (843, 256)  │  1.5s

──────────────────────────────────────────────────────────────────────
  Démarrage des expériences (6 valeurs de N × 5 runs)
  Bio-JEPA : 200 époques (sonde)  │  GNN sup. : 300 époques
──────────────────────────────────────────────────────────────────────

  ── N = 10 ──────────────────────────────────────────────
    Bio-JEPA  N=   10  run 1/5  →  r=-0.1195  ρ=-0.1220  RMSE=4.0249
    Bio-JEPA  N=   10  run 2/5  →  r=+0.0138  ρ=+0.0566  RMSE=3.4711
    Bio-JEPA  N=   10  run 3/5  →  r=+0.076

---
## Étape 2 — Tableau comparatif : A2A / EGFR / Mpro

In [6]:
import json, pandas as pd

with open('results/few_shot_transfer.json') as f:
    a2a = json.load(f)
with open('results/few_shot_egfr.json') as f:
    egfr = json.load(f)
with open('results/few_shot_mpro.json') as f:
    mpro = json.load(f)

def get_r(data, i):
    entry = data['bio_jepa'][i]
    if 'mean_r' in entry:
        return round(entry['mean_r'], 3)
    return round(entry.get('pearson_r', {}).get('mean', 0), 3)

N = a2a['N_values']
rows = []
for i, n in enumerate(N):
    rows.append({
        'N': n,
        'A2A (ChEMBL251)':   get_r(a2a, i),
        'EGFR (ChEMBL203)':  get_r(egfr, i),
        'Mpro (ChEMBL325)':  get_r(mpro, i),
    })

df = pd.DataFrame(rows)
print('=== BIO-JEPA FEW-SHOT — Pearson r sur 3 cibles ===')
print(df.to_string(index=False))

# Résumé
print('\n=== RÉSUMÉ @ N=1000 ===')
for col in ['A2A (ChEMBL251)', 'EGFR (ChEMBL203)', 'Mpro (ChEMBL325)']:
    print(f'  {col}: r = {df[col].iloc[-1]}')

=== BIO-JEPA FEW-SHOT — Pearson r sur 3 cibles ===
   N  A2A (ChEMBL251)  EGFR (ChEMBL203)  Mpro (ChEMBL325)
  10            0.036             0.003             0.024
  50            0.130             0.130             0.130
 100            0.174             0.174             0.174
 200            0.278             0.277             0.277
 500            0.465             0.465             0.465
1000            0.542             0.539             0.540

=== RÉSUMÉ @ N=1000 ===
  A2A (ChEMBL251): r = 0.542
  EGFR (ChEMBL203): r = 0.539
  Mpro (ChEMBL325): r = 0.54


---
## Étape 3 — Sauvegarde Drive

In [7]:
shutil.copy('results/few_shot_mpro.json', f'{DRIVE_RES}/few_shot_mpro.json')
print('✓ few_shot_mpro.json sauvegardé sur Drive')
print('✅ NOTEBOOK 2 TERMINÉ')

✓ few_shot_mpro.json sauvegardé sur Drive
✅ NOTEBOOK 2 TERMINÉ
